# Preparación, limpieza y estructuración de datos: Instacart

**Actividad:** GA1-220501094-01-AA1-EV07 · Preparación, limpieza y estructuración de datos del proyecto individual con Pandas
**Aprendiz:** Carlos Eduardo Herrera Sibaja
**Programa:** Análisis y Desarrollo de Software
**Fecha:** 24/11/2026

**Pregunta analítica:** ¿Qué patrones descriptivos se observan en los pedidos y artículos de Instacart, en particular en el horario de compra y la recompra de productos?

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos para obtener información confiable que permita responder la pregunta analítica?

Este cuaderno parte del diagnóstico de `notebooks/data_quality.ipynb` y `reports/data_quality.md`. Está organizado en tres partes: limpieza y estandarización (Lección 2), filtrado, ordenamiento y agrupación (Lección 3) y combinación y estructuración (Lección 4).

**Enfoque propio del proyecto.** El análisis no repite los laboratorios ni los retos de Instacart (histogramas, listas de los 20 productos más pedidos, relleno con valores centinela). Se centra en la pregunta del proyecto: cómo cambian la demanda, el tamaño de la canasta y la recompra según la **franja horaria**, el **fin de semana**, la **etapa del cliente** y el **intervalo entre compras**. Los datos originales en `data/raw/` no se modifican; los resultados se guardan en `data/processed/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
if not RAW_DIR.exists():  # ejecución desde la raíz del proyecto
    RAW_DIR, PROCESSED_DIR = Path('data/raw'), Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVOS = {
    'orders': 'instacart_orders.csv',
    'order_products': 'order_products.csv',
    'products': 'products.csv',
    'aisles': 'aisles.csv',
    'departments': 'departments.csv',
}
raw = {nombre: pd.read_csv(RAW_DIR / archivo, sep=';') for nombre, archivo in ARCHIVOS.items()}
for nombre, df in raw.items():
    print(f'{nombre:<15} {df.shape[0]:>10,} filas x {df.shape[1]} columnas')

orders             478,967 filas x 6 columnas
order_products   4,545,007 filas x 4 columnas
products            49,694 filas x 4 columnas
aisles                 134 filas x 2 columnas
departments             21 filas x 2 columnas


## 1. Limpieza y estandarización (Lección 2)

Los problemas que se tratan provienen del diagnóstico de calidad:

| Dataset | Problema del diagnóstico | Tratamiento aplicado | Justificación |
|---|---|---|---|
| `orders` | 15 filas duplicadas exactas | Eliminar duplicados | Son el mismo pedido repetido; inflarían los conteos por hora y día |
| `orders` | 28.819 nulos en `days_since_prior_order` | Conservar como nulo (`Int8`) y crear `is_first_order` | Son primeros pedidos: no existe un pedido anterior. Imputar 0 inventaría una recompra inmediata |
| `order_products` | 836 nulos en `add_to_cart_order` | Conservar como nulo (`Int8`) | La posición no se registró a partir del artículo 65; las filas sí sirven para medir la recompra |
| `products` | 1.258 nombres nulos | Rellenar con `unknown product` | Todos pertenecen al pasillo y departamento `missing`; la etiqueta evita nulos al agrupar sin borrar productos que aparecen en pedidos |
| `products` | Nombres con espacios sobrantes y mayúsculas distintas | Normalizar texto y crear `canonical_product_id` | El mismo producto aparece con ids distintos; el id canónico permite agruparlos sin perder el id original |
| `aisles`, `departments` | Categoría `missing` | Renombrar a `unclassified` | Estandariza el nombre de la categoría sin clasificar |
| Todos | Tipos `int64`/`float64` genéricos | Tipos ajustados (`int8`–`int32`, `Int8`, `bool`, `category`) | Tipos coherentes con el significado y menor uso de memoria |

### 1.1 Estado inicial de los datos
La función `perfil_calidad()` resume las métricas que se comparan antes y después del procesamiento.

In [2]:
def perfil_calidad(df):
    # Métricas de calidad que se comparan antes y después de la limpieza.
    return {
        'filas': len(df),
        'columnas': df.shape[1],
        'duplicados_exactos': int(df.duplicated().sum()),
        'valores_nulos': int(df.isna().sum().sum()),
        'columnas_con_nulos': int(df.isna().any().sum()),
        'memoria_MB': round(df.memory_usage(deep=True).sum() / 1024**2, 1),
    }


calidad_inicial = pd.DataFrame.from_dict({nombre: perfil_calidad(df) for nombre, df in raw.items()}, orient='index')
calidad_inicial

,filas,columnas,duplicados_exactos,valores_nulos,columnas_con_nulos,memoria_MB
orders,478967,6,15,28819,1,21.9
order_products,4545007,4,0,836,1,138.7
products,49694,4,0,1258,1,2.9
aisles,134,2,0,0,0,0.0
departments,21,2,0,0,0,0.0


In [3]:
nombres = raw['products']['product_name'].dropna()
nombres_normalizados = nombres.str.strip().str.replace(r'\s+', ' ', regex=True).str.lower()
print('Nombres de producto con espacios sobrantes:',
      int((nombres != nombres.str.strip().str.replace(r'\s+', ' ', regex=True)).sum()))
print('Nombres repetidos solo por mayúsculas o espacios:', int(nombres_normalizados.duplicated().sum()))
print('Pedidos con order_number == 1 y days_since_prior_order nulo:',
      int((raw['orders']['order_number'].eq(1) & raw['orders']['days_since_prior_order'].isna()).sum()))

Nombres de producto con espacios sobrantes: 419
Nombres repetidos solo por mayúsculas o espacios: 141
Pedidos con order_number == 1 y days_since_prior_order nulo: 28819


### 1.2 Funciones reutilizables de limpieza
Cada función recibe el DataFrame como primer argumento y lo devuelve transformado, por lo que pueden encadenarse con `pipe()`. Todas anotan el paso en `registro_cambios`, que sirve como bitácora del procesamiento.

In [4]:
registro_cambios = []


def registrar(df, dataset, paso, detalle=''):
    # Anota el estado del DataFrame tras un paso y lo devuelve sin cambios (apto para pipe).
    registro_cambios.append({
        'dataset': dataset,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'detalle': detalle,
    })
    return df


def eliminar_duplicados(df, dataset, subset=None):
    filas_antes = len(df)
    df = df.drop_duplicates(subset=subset, ignore_index=True)
    return registrar(df, dataset, 'eliminar_duplicados', f'{filas_antes - len(df)} filas eliminadas')


def normalizar_texto(df, dataset, columnas):
    # Minúsculas, sin espacios al inicio o al final y sin espacios repetidos.
    df = df.copy()
    modificados = 0
    for col in columnas:
        limpio = df[col].str.strip().str.replace(r'\s+', ' ', regex=True).str.lower()
        modificados += int((limpio != df[col]).fillna(False).sum())
        df[col] = limpio
    return registrar(df, dataset, 'normalizar_texto', f'{", ".join(columnas)}: {modificados} valores modificados')


def rellenar_nulos(df, dataset, valores):
    detalle = ', '.join(f'{col}: {int(df[col].isna().sum())} nulos → {valor!r}' for col, valor in valores.items())
    return registrar(df.fillna(valores), dataset, 'rellenar_nulos', detalle)


def reemplazar_valores(df, dataset, columna, reemplazos):
    afectados = int(df[columna].isin(list(reemplazos)).sum())
    df = df.assign(**{columna: df[columna].replace(reemplazos)})
    return registrar(df, dataset, 'reemplazar_valores', f'{columna}: {afectados} valores {reemplazos}')


def convertir_tipos(df, dataset, tipos):
    df = df.astype(tipos)
    return registrar(df, dataset, 'convertir_tipos', ', '.join(f'{col} → {tipo}' for col, tipo in tipos.items()))

### 1.3 Pipeline de limpieza: pedidos (`orders`)
Se eliminan los duplicados, se ajustan los tipos y se agregan dos columnas: `is_first_order`, que explica los nulos del intervalo, y `day_name`, que estandariza el día de la semana. Se asume la convención habitual de Instacart: `order_dow = 0` es domingo.

In [5]:
DIAS = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles', 4: 'jueves', 5: 'viernes', 6: 'sábado'}


def marcar_primer_pedido(df, dataset):
    df = df.assign(is_first_order=df['order_number'].eq(1))
    return registrar(df, dataset, 'agregar_columna', 'is_first_order = order_number == 1')


def agregar_dia_semana(df, dataset):
    dia = pd.Categorical(df['order_dow'].map(DIAS), categories=list(DIAS.values()), ordered=True)
    return registrar(df.assign(day_name=dia), dataset, 'agregar_columna', 'day_name a partir de order_dow (0 = domingo)')


orders_clean = (
    raw['orders']
    .pipe(registrar, 'orders', 'carga_inicial')
    .pipe(eliminar_duplicados, 'orders')
    .pipe(convertir_tipos, 'orders', {
        'order_id': 'int32',
        'user_id': 'int32',
        'order_number': 'int16',
        'order_dow': 'int8',
        'order_hour_of_day': 'int8',
        'days_since_prior_order': 'Int8',
    })
    .pipe(marcar_primer_pedido, 'orders')
    .pipe(agregar_dia_semana, 'orders')
)
orders_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 478952 entries, 0 to 478951
Data columns (total 8 columns):
 #   Column                  Non-Null Count   Dtype   
---  ------                  --------------   -----   
 0   order_id                478952 non-null  int32   
 1   user_id                 478952 non-null  int32   
 2   order_number            478952 non-null  int16   
 3   order_dow               478952 non-null  int8    
 4   order_hour_of_day       478952 non-null  int8    
 5   days_since_prior_order  450135 non-null  Int8    
 6   is_first_order          478952 non-null  bool    
 7   day_name                478952 non-null  category
dtypes: Int8(1), bool(1), category(1), int16(1), int32(2), int8(2)
memory usage: 7.3 MB


### 1.4 Pipeline de limpieza: artículos por pedido (`order_products`)
Se comprueba que no haya un mismo producto repetido dentro de un pedido, y se ajustan los tipos. `add_to_cart_order` conserva sus nulos con el tipo entero con nulos `Int8`, y `reordered` pasa a booleano.

In [6]:
order_products_clean = (
    raw['order_products']
    .pipe(registrar, 'order_products', 'carga_inicial')
    .pipe(eliminar_duplicados, 'order_products', subset=['order_id', 'product_id'])
    .pipe(convertir_tipos, 'order_products', {
        'order_id': 'int32',
        'product_id': 'int32',
        'add_to_cart_order': 'Int8',
        'reordered': 'bool',
    })
)
order_products_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4545007 entries, 0 to 4545006
Data columns (total 4 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int32
 1   product_id         int32
 2   add_to_cart_order  Int8 
 3   reordered          bool 
dtypes: Int8(1), bool(1), int32(2)
memory usage: 47.7 MB


### 1.5 Pipeline de limpieza: productos y catálogos
Los nombres se normalizan antes de rellenar los nulos, para que la etiqueta `unknown product` no se mezcle con nombres reales. `unificar_productos()` asigna a los productos con el mismo nombre normalizado el menor `product_id` como identificador canónico; el `product_id` original se conserva porque es el que usan los pedidos.

In [7]:
NOMBRE_DESCONOCIDO = 'unknown product'
CATEGORIA_SIN_CLASIFICAR = 'unclassified'


def unificar_productos(df, dataset):
    canonico = df.groupby('product_name')['product_id'].transform('min')
    canonico = canonico.where(df['product_name'] != NOMBRE_DESCONOCIDO, df['product_id'])
    df = df.assign(canonical_product_id=canonico)
    unificados = int((df['canonical_product_id'] != df['product_id']).sum())
    return registrar(df, dataset, 'unificar_productos', f'{unificados} productos apuntan a un id canónico')


products_clean = (
    raw['products']
    .pipe(registrar, 'products', 'carga_inicial')
    .pipe(eliminar_duplicados, 'products')
    .pipe(normalizar_texto, 'products', ['product_name'])
    .pipe(rellenar_nulos, 'products', {'product_name': NOMBRE_DESCONOCIDO})
    .pipe(unificar_productos, 'products')
    .pipe(convertir_tipos, 'products', {
        'product_id': 'int32',
        'canonical_product_id': 'int32',
        'aisle_id': 'int16',
        'department_id': 'int8',
    })
)

aisles_clean = (
    raw['aisles']
    .pipe(registrar, 'aisles', 'carga_inicial')
    .pipe(normalizar_texto, 'aisles', ['aisle'])
    .pipe(reemplazar_valores, 'aisles', 'aisle', {'missing': CATEGORIA_SIN_CLASIFICAR})
    .pipe(convertir_tipos, 'aisles', {'aisle_id': 'int16'})
)

departments_clean = (
    raw['departments']
    .pipe(registrar, 'departments', 'carga_inicial')
    .pipe(normalizar_texto, 'departments', ['department'])
    .pipe(reemplazar_valores, 'departments', 'department', {'missing': CATEGORIA_SIN_CLASIFICAR})
    .pipe(convertir_tipos, 'departments', {'department_id': 'int8'})
)

duplicados_por_nombre = products_clean[products_clean['canonical_product_id'] != products_clean['product_id']]
products_clean[products_clean['canonical_product_id'].isin(duplicados_por_nombre['canonical_product_id'])] \
    .sort_values(['canonical_product_id', 'product_id']).head(6)

,product_id,product_name,aisle_id,department_id,canonical_product_id
41,42,biotin 1000 mcg,47,11,42
2058,2059,biotin 1000 mcg,47,11,42
185,186,fresh scent dishwasher detergent with dawn,74,17,186
19042,19043,fresh scent dishwasher detergent with dawn,74,17,186
515,516,american cheese slices,21,16,516
9037,9038,american cheese slices,21,16,516


### 1.6 Validaciones de la limpieza
Cada `assert` detiene la ejecución si una regla no se cumple, de modo que los pasos siguientes solo se ejecutan con datos válidos.

In [8]:
# Pedidos
assert orders_clean['order_id'].is_unique, 'order_id debe ser único'
assert len(orders_clean) == len(raw['orders'].drop_duplicates())
assert orders_clean['order_hour_of_day'].between(0, 23).all()
assert orders_clean['order_dow'].between(0, 6).all()
assert orders_clean['days_since_prior_order'].dropna().between(0, 30).all()
assert (orders_clean['days_since_prior_order'].isna() == orders_clean['is_first_order']).all(), \
    'los nulos del intervalo deben ser exactamente los primeros pedidos'

# Artículos por pedido
assert not order_products_clean.duplicated(['order_id', 'product_id']).any()
assert order_products_clean['add_to_cart_order'].dropna().between(1, 64).all()
assert order_products_clean['reordered'].dtype == bool

# Productos y catálogos
assert products_clean['product_id'].is_unique
assert products_clean['product_name'].notna().all()
assert not products_clean['product_name'].str.contains(r'^\s|\s$|\s{2,}', regex=True).any()
assert products_clean.groupby('canonical_product_id')['product_name'].nunique().eq(1).all()
assert aisles_clean['aisle_id'].is_unique and departments_clean['department_id'].is_unique
assert 'missing' not in set(aisles_clean['aisle']) and 'missing' not in set(departments_clean['department'])

print('Todas las validaciones de limpieza se cumplen.')

Todas las validaciones de limpieza se cumplen.


### 1.7 Registro de cambios y comparación antes/después

In [9]:
pd.DataFrame(registro_cambios)

,dataset,paso,filas,nulos,detalle
0,orders,carga_inicial,478967,28819,
1,orders,eliminar_duplicados,478952,28817,15 filas eliminadas
2,orders,convertir_tipos,478952,28817,"order_id → int32, user_id → int32, order_numbe..."
3,orders,agregar_columna,478952,28817,is_first_order = order_number == 1
4,orders,agregar_columna,478952,28817,day_name a partir de order_dow (0 = domingo)
5,order_products,carga_inicial,4545007,836,
6,order_products,eliminar_duplicados,4545007,836,0 filas eliminadas
7,order_products,convertir_tipos,4545007,836,"order_id → int32, product_id → int32, add_to_c..."
8,products,carga_inicial,49694,1258,
9,products,eliminar_duplicados,49694,1258,0 filas eliminadas


In [10]:
limpios = {
    'orders': orders_clean,
    'order_products': order_products_clean,
    'products': products_clean,
    'aisles': aisles_clean,
    'departments': departments_clean,
}
calidad_limpia = pd.DataFrame.from_dict({nombre: perfil_calidad(df) for nombre, df in limpios.items()}, orient='index')

metricas = ['filas', 'duplicados_exactos', 'valores_nulos', 'memoria_MB']
comparacion_limpieza = pd.concat({'antes': calidad_inicial, 'después': calidad_limpia}, axis=1).swaplevel(axis=1)[metricas]
comparacion_limpieza

filas          duplicados_exactos         valores_nulos  \
                  antes  después              antes después         antes   
orders           478967   478952                 15       0         28819   
order_products  4545007  4545007                  0       0           836   
products          49694    49694                  0       0          1258   
aisles              134      134                  0       0             0   
departments          21       21                  0       0             0   

                       memoria_MB          
               después      antes después  
orders           28817       21.9     7.3  
order_products     836      138.7    47.7  
products             0        2.9     2.3  
aisles               0        0.0     0.0  
departments          0        0.0     0.0

**Interpretación.** Se eliminaron las 15 filas duplicadas de pedidos (de 478.967 a 478.952). Los nulos de `orders` bajan de 28.819 a 28.817 porque dos de los duplicados eran primeros pedidos. Los nulos que quedan son exactamente los primeros pedidos, como confirma el `assert`, y se conservan a propósito, igual que los 836 de `add_to_cart_order`. En `products` desaparecen los 1.258 nulos. La normalización corrige 419 nombres con espacios sobrantes y detecta 141 productos cuyo nombre repite el de otro. El diagnóstico contaba 104 porque no unificaba los espacios internos. Con el ajuste de tipos, la memoria de `order_products` baja de 138,7 MB a 47,7 MB y la de `orders`, de 21,9 MB a 7,3 MB.